## Setup

In [4]:
import torch

## 01 Attention Exploration (14 points)

Multi-head self-attention is the core modeling component of Transformers. In this question, we’ll get some practice working with the self-attention equations, and motivate why multi-headed self-attention can be preferable to single-headed self-attention.

Recall that attention can be viewed as an operation on a *query* vector $q \in \mathbb{R}^d$, a set of *value* vectors
$\{v_1,...,v_n\},v_i \in \mathbb{R}^d$, and a set of *key* vectors $\{k_1,...,k_n\},k_i \in \mathbb{R}^d$, specified as follows:

$$
c = \sum_{i=1}^{n} v_i \alpha_i \quad (1)
$$

$$
\alpha_i =
\frac{\exp(k_i^\top q)}
{\sum_{j=1}^{n} \exp(k_j^\top q)} \quad (2)
$$

with $\alpha= {\alpha_1,...,\alpha_n}$ termed the "attention weights". Observe that the output $c\in\mathbb{R}^d$ is an average over the value vectors weighted with respect to $\alpha$.

### (a) (3 points) Copying in attention.

One advantage of attention is that it’s particularly easy to "copy" a value vector to the output c. In this problem, we’ll motivate why this is the case.

i. (2 points) The distribution $\alpha$ is typically relatively "diffuse"; the probability mass is spread out between many different $\alpha_i$. However, this is not always the case. Describe (in one sentence) under what conditions the categorical distribution $\alpha$ puts almost all of its weight on some $\alpha_j$, where $j \in\{1,...,n\}$ (i.e. $\alpha_j \gg \sum_{i\not=j}\alpha_i$). What must be true about the query $q$ and/or the keys $\{k_1,...,k_n\}$?


Here's the main idea. Suppose $k_i$ is just the set of unit orthonormal vectors. Suppose we have a one-hot-vector with 1 in the $j$-th position and 0 elsewhere. This does not quite work due to the fact that $exp(0) = 1$, but we may put some large constant $C \gg 1$ in the $j$-th position instead. The general answer follows.

The attention distribution puts almost all its weight on $\alpha_j$ when the query $q$ has a much larger dot product with key $k_j$ than with any other key; that is, $k_j^\top q \gg k_i^\top q$ for all $i\ne j$. In this case, $\exp(k_j^\top q)$ dominates the denominator of the softmax, so $\alpha_j\approx 1$ and $\alpha_i\approx 0$ for $i\ne j$. This can happen when $q$ and $k_j$ are strongly aligned while the other keys are not.

---


ii. (1 point) Under the conditions you gave in (i), describe the output c.

In this case $c \approx v_j$ since $\alpha_j \approx 1$ and $\alpha_i \approx 0$ for all $i \ne j$.

### (b) (2 points) An average of two.

Instead of focusing on just one vector $v_j$, a Transformer model might want to incorporate information from multiple source vectors. Consider the case where we instead want to incorporate information from two vectors $v_a$ and $v_b$, with corresponding key vectors $k_a$ and $k_b$. Assume that (1) all key vectors are orthogonal, so $k_i^\top k_j = 0$ for all $i \ne j$; and (2) all key vectors have norm 1. Find an expression for a query vector $q$ such that $c \approx \frac{1}{2} (v_a + v_b)$, and justify your answer. (Recall what you learned in part (a).)  
Hint: while the softmax function will never exactly average the two vectors, you can get close by using a large scalar
multiple in the expression.


WLOG, assume that $k_i$ are just unit vectors with $k_{ii} = 1$ and $k_{ij} = 0$ for all $i \ne j$. Then we may take $q_a = q_b = C$, where $C$ is some large number and the other components of $q$ are 0. This ensures that the attention weights for $v_a$ and $v_b$ are approximately equal and dominate the softmax, giving $c \approx \frac{1}{2} (v_a + v_b)$.

#### Answer from LUNA

Your answer is essentially correct. The cleanest expression is

$$
q=C(k_a+k_b),
$$

where $C>0$ is large. Then, using orthonormal keys,

$$
k_a^\top q=C,\qquad k_b^\top q=C,
$$

while every other key has score $0$. Therefore,

$$
\alpha_a=\alpha_b
=\frac{e^C}{2e^C+(n-2)}
\longrightarrow \frac12
$$

as $C\to\infty$, and the remaining attention weights approach $0$. Hence,

$$
c\approx \frac12v_a+\frac12v_b.
$$

Your coordinate version is valid if the keys are identified with the standard basis, but change $k_{ii}$ and $k_{ij}$ to statements about the **components** of $k_i$, and use $a,b$ consistently instead of $i,j$.

### (c) (5 points) Drawbacks of single-headed attention

In the previous part, we saw how it was possible for a single-headed attention to focus equally on two values. The same concept could easily be extended to any subset of values. In this question we’ll see why it’s not a practical solution.

Consider a set of key vectors $\{k_1,...,k_n\}$ that are now randomly sampled, $k_i \sim N(\mu_i,\Sigma_i)$, where the means $\mu_i \in \mathbb{R}^d$ are known to you, but the covariances $\Sigma_i$ are unknown (unless specified otherwise in the question). Further, assume that the means $\mu_i$ are all perpendicular; $\mu_i^\top \mu_j = 0$ if $i\neq j$, and unit norm, $\|\mu_i\|= 1$.

---

i. (2 points) Assume that the covariance matrices are $\Sigma_i = \alpha I, \forall i \in \{1,2,...,n\}$, for vanishingly small $\alpha$. Design a query $q$ in terms of the $\mu_i$ such that as before, $c\approx\frac{1}{2} (v_a+ v_b)$, and provide a brief argument as to why it works.

My initial idea was to sample the query from $q_a \sim N(\mu_a,\Sigma_i)$ and $q_b \sim N(\mu_b,\Sigma_i)$ and then take a sum $q = C (q_a + q_b)$. And this would work since with a small $\alpha$ this $q$ will be approximately orthogonal to $k_i$ for all $i \notin \{a,b\}$.

But in the exercise they ask about $q$ in terms of the means $\mu_a$ and $\mu_b$, actually. SO instead of sampleing we may jjust take a sum of these means: $q = C (\mu_a + \mu_b)$, where $C>0$ is large. This works for the same reason as before: $q$ will have a large inner product with $k_a$ and $k_b$, and approximately zero with all other $k_i$.

---

ii. (3 points) Though single-headed attention is resistant to small perturbations in the keys, some types of larger perturbations may pose a bigger issue. In some cases, one key vector ka may be larger or smaller in norm than the others, while still pointing in the same direction as $\mu_a$.

As an example, let us consider a covariance for item a as $\Sigma_a = \alpha I+ \frac{1}{2} (\mu_a\mu_a^\top)$ for vanishingly small $\alpha$ (as shown in figure 1). This causes $k_a$ to point in roughly the same direction as $\mu_a$, but with large variances in magnitude. Further, let $\Sigma_i = \alpha I$ for all $i\not= a$.

When you sample $\{k_1,...,k_n\}$ multiple times, and use the $q$ vector that you defined in part i.,
what do you expect the vector $c$ will look like qualitatively for different samples? Think about
how it differs from part (i) and how $c$’s variance would be affected.

So again, $q = C (\mu_a + \mu_b)$. We still have $k_i^T q = 0$ for all $i \notin \{a,b\}$. We also still have $k_b^T q = C k_b^T (\mu_a + \mu_b) = C k_b^T \mu_b = C \|\mu_b\|^2 = C$ since $k_b$ is approximately aligned with $\mu_b$ due to small variance and has the same norm as $\mu_b$, which is 1.

But what about $k_a^T q$? Since $k_a$ may have a larger or smaller norm than $\mu_a$ while still pointing in roughly the same direction, $k_a^T q = C k_a^T (\mu_a + \mu_b) \approx C k_a^T \mu_a = C \|k_a\| \|\mu_a\| \cos\theta \approx C \|k_a\|$, where $\theta$ is the small angle between $k_a$ and $\mu_a$. Therefore, $k_a^T q$ could vary significantly depending on the sampled magnitude of $k_a$.

#### Answer from LUNA

Yes, your main reasoning is correct: the perturbation in $k_a$ changes its **norm in the $\mu_a$ direction**, so the score $k_a^\top q$ can vary substantially even though $q$ remains fixed.

A few corrections:

1. For sampled keys, $k_i^\top q$ is only approximately zero for $i\notin\{a,b\}$.
2. $k_b^\top q$ is approximately $C$, not exactly $C$, because $k_b$ also has small noise.
3. The important issue with $k_a$ is best expressed through its projection onto $\mu_a$:

$$
k_a \approx (1+z)\mu_a,
\qquad z\sim N\left(0,\frac12\right),
$$

ignoring the vanishingly small orthogonal noise. Therefore,

$$
k_a^\top q
\approx C(1+z),
$$

whereas

$$
k_b^\top q\approx C.
$$

Thus, depending on the sample:

- if $z>0$, $k_a^\top q$ may exceed $k_b^\top q$, so $c$ moves toward $v_a$;
- if $z<0$, $k_b^\top q$ may dominate, so $c$ moves toward $v_b$;
- if $z\approx0$, the output is close to $\frac12(v_a+v_b)$.

Because the softmax exponentially amplifies score differences, $c$ can vary greatly across samples. Consequently, compared with part (i), the output has substantially larger variance and may switch between values near $v_a$, values near $v_b$, and intermediate mixtures.

Your geometric explanation is good, but saying that $k_a$ “points in roughly the same direction” can be misleading when its longitudinal coefficient $1+z$ becomes negative or close to zero. The projection formulation above is more precise.

### (d) (3 points) Benefits of multi-headed attention

Now we’ll see some of the power of multi-headed attention. We’ll consider a simple version of multi-headed attention which is identical to single-headed self-attention as we’ve presented it, except two query vectors ($q_1$ and $q_2$) are defined, which leads to a pair of vectors ($c_1$ and $c_2$), each the output of single-headed attention given its respective query vector. The final output of the multi-headed attention is their average, $1/2 (c_1 + c_2)$.

As in question 1(c), consider a set of key vectors ${k_1,...,k_n}$ that are randomly sampled, $k_i ∼ N(µ_i,Σ_i)$, where the means $µ_i$ are known to you, but the covariances $Σ_i$ are unknown. Also as before, assume that the means $µ_i$ are mutually orthogonal; $µ_i^\top µ_j = 0$ if $i \ne j$, and unit norm, $\|µ_i\|= 1$.

---

i. (1 point) Assume that the covariance matrices are $Σ_i = αI$, for vanishingly small $\alpha$. Design $q_1$ and $q_2$ in terms of $µ_i$ such that $c$ is approximately equal to $\frac{1}{2} (v_a + v_b)$. Note that $q_1$ and $q_2$ should have different expressions

As above let's take $q_1 = C µ_a, \quad q_2 = C µ_b$, where $C$ is a large constant $C \gg 1$. All $k_i$ are approximately equal to their means $µ_i$ due to the vanishingly small covariance, so $k_i^T q_1 \approx C µ_i^T µ_a = C \delta_{ia}$ and $k_i^T q_2 \approx C µ_i^T µ_b = C \delta_{ib}$, where $\delta_{ij}$ is the Kronecker delta. 

Therefore, the attention weights will be approximately one for the corresponding keys and zero for all others, leading to $c_1 \approx v_a$ and $c_2 \approx v_b$, and thus $c \approx \frac{1}{2} (v_a + v_b)$.

---

ii. (2 points) Assume that the covariance matrices are $Σ_a = \alpha I+ \frac{1}{2} (\mu_a \mu_a^⊤)$ for vanishingly small
$\alpha$, and $\Sigma_i = \alpha I$ for all $i \neq a$. Take the query vectors $q_1$ and $q_2$ that you designed in part
i. What, qualitatively, do you expect the output c to look like across different samples of the key vectors? Explain briefly in terms of variance in $c_1$ and $c_2$. You can ignore cases in which $k_a^⊤ q_i <0$.

First of all, we still have $c_2 \approx v_b$. 

What about $c_1$? Well, the direction of $k_a$ is still the same as $\mu_a$, so orthogonality will hold and we get 

$$k_a^T q_1 \approx C k_a^T µ_a = C \|k_a\| \|\mu_a\| cos \theta \approx C \|k_a\| = s_a$$

where $\theta$ is the (small) angle between $k_a$ and $\mu_a$. Key condition: If score $s_a$ is still large enough, we will get the same result as in part i, namely $c_1 \approx v_a$.

---

### (e) multi-headed attention advantages

(1 point) Based on part (d), briefly summarize how multi-headed attention overcomes the drawbacks of single-headed attention that you identified in part (c).

A single-headed attention mechanism can *in principle* focus on two or more aspects of the input, but they are not independent and may interfere with each other. One of them can be skewed or dominated by the other, leading to suboptimal attention distribution.

Multi-headed attention, on the other hand, allows the model to attend to different parts of the input *in parallel*, *independently*, thereby overcoming this limitation.

#### Answer from LUNA

Your answer is correct in general, but it would be stronger if it explicitly connected to part (d): a perturbation in one key can distort a single head’s combined attention, whereas it affects mainly one multi-head component.

A polished version:

> Single-headed attention must use one query and one softmax distribution to attend to multiple values. Consequently, a perturbation in one key can change the relative attention weights and interfere with attention to the other values. Multi-headed attention uses separate queries and softmax distributions, allowing different heads to focus on different values independently. Thus, a perturbation affecting one head has much less effect on the other head, making the final averaged output more robust.

This directly explains the benefit shown in the preceding parts.

## 02 Position Embeddings Exploration (6 points)

Position embeddings are an important component of the Transformer architecture, allowing the model to differentiate between tokens based on their position in the sequence. In this question, we’ll explore the need for positional embeddings in Transformers and how they can be designed.

Recall that the crucial components of the Transformer architecture are the self-attention layer and the
feed-forward neural network layer. Given an input tensor $X ∈ \mathbb{R}^{T×d}$, where T is the sequence length
and d is the hidden dimension, the self-attention layer computes the following:

$$ Q= XW_Q,\quad K= XW_K,\quad V= XW_V $$

$$ H = \text{softmax}\left(\frac{QK^\top}{\sqrt{d}}\right)V $$

where $W_Q, W_K, W_V ∈ \mathbb{R}^{d×d}$ are weight matrices, and $H ∈ \mathbb{R}^{T×d}$ is the output.

Next, the feed-forward layer applies the following transformation:

$$ Z = \text{ReLU}(HW_1 + \mathbf{1}·b_1)W_2 + \mathbf{1}·b_2 $$

where $W_1,W_2 \in \mathbb{R}^{d×d}$ and $b_1,b_2 ∈ \mathbb{R}^{1×d}$ are weights and biases; $\mathbf{1} \in \mathbb{R}^{T×1}$ is a vector of ones‡; and $Z \in \mathbb{R}^{T×d}$ is the final output.


### (a) (4 points) Permuting the input.

i. (3 points) Suppose we permute the input sequence X such that the tokens are shuffled randomly. This can be represented as multiplication by a permutation matrix $P \in \mathbb{R}^{T×T}$, i.e. $X_{\text{perm}} = PX$. (See Wikipedia for a recap on permutation matrices.)

Show that the output $Z_{\text{perm}}$ for the permuted input $X_{\text{perm}}$ will be $Z_{\text{perm}} = PZ$. You are given that for any permutation matrix $P$ and any matrix $A$, the following hold:
$\text{softmax}(PAP^\top) = P \text{softmax}(A) P^\top$ and $\text{ReLU}(PA) = P \text{ReLU}(A)$.

Let's first show that $H_{\text{perm}} = P H$. We have the following:

$$Q_{\text{perm}} K_{\text{perm}} ^ T = P X W_Q W_K^T X^T P^T = P Q K ^ T P^T$$

$$
\begin{aligned}
H_{\text{perm}}
&= \text{softmax}\left(\frac{Q_{\text{perm}} K_{\text{perm}}^T}{\sqrt{d}}\right)V_{\text{perm}} \\
&= \text{softmax}\left(\frac{P Q K^T P^T}{\sqrt{d}}\right)PV \\
&= P\,\text{softmax}\left(\frac{Q K^T}{\sqrt{d}}\right) P^T P V \\
&= P\,\text{softmax}\left(\frac{Q K^T}{\sqrt{d}}\right)V \\
&= PH
\end{aligned}
$$

since $P^T P = I$, the identity matrix, which allows us to simplify the expression in the fourth line.

$$
\begin{aligned}

Z_{\text{perm}} 
&= \text{ReLU}(P H W_1 + P \mathbf{1}·b_1)W_2 + P \mathbf{1}·b_2  \\
&= P \text{ReLU}(H W_1 + \mathbf{1}·b_1)W_2 + P \mathbf{1}·b_2  \\
&= P Z

\end{aligned}
$$

since $ P \mathbf{1}·b_1 = \mathbf{1}·b_1$ and $ P \mathbf{1}·b_2 = \mathbf{1}·b_2$ for a permutation matrix $P$.

ii. (1 point) Think about the implications of the result you derived in part i. Explain why this property of the Transformer model could be problematic when processing text.

This result means that the Transformer get the same output for any permutation of the input sequence. In other words, it doesn't inherently capture the order of the tokens in the sequence.

Almost. The result says the output is **permuted along with the input**, not that it is numerically unchanged. A more precise version is:

> This result means that permuting the input sequence simply permutes the Transformer’s output in the same way. Without positional information, the Transformer does not inherently distinguish token order, which can be problematic when order matters in text.

### (b) (2 points) Position embeddings

(2 points) Position embeddings are vectors that encode the position of each token in the sequence. They are added to the input word embeddings before feeding them into the Transformer. 

One approach is to generate position embedding using a fixed function of the position and the dimension of the embedding. If the input word embeddings are $X \in \mathbb{R}^{T \times d}$, the position embeddings $\Phi \in \mathbb{R}^{T \times d}$ are generated as follows:

$$\Phi_{(t,2i)} = \sin \frac{t}{10000^{2i/d}}$$

$$\Phi_{(t,2i+1)} = \cos \frac{t}{10000^{2i/d}}$$

where $t \in \{0,1,...,T-1\}$ and $i \in \{0,1,...,d/2-1\}$.

Specifically, the position embeddings are added to the input word embeddings:

$$X_{\text{pos}} = X + \Phi$$

i. (1 point) Do you think the position embeddings will help the issue you identified in part (a)? If yes, explain how and if not, explain why not.

ii. (1 point) Can the position embeddings for two different tokens in the input sequence be the
same? If yes, provide an example. If not, explain why not.

First of all, what does this mean in terms of $\Phi$? Can two distinct positions have identical embedding vectors? 
That is, can $\Phi_{t_1} = \Phi_{t_2}$ (meaning $\Phi_{(t_1, j)} = \Phi_{(t_2, j)}$ for all $j \in \{0, \dots, d-1\}$)? Answer: No, they cannot be the same.

Look at just the very first pair of dimensions ($i = 0$):
$$\omega_0 = \frac{1}{10000^0} = 1$$
$$\Phi_{(t, 0)} = \sin(t), \qquad \Phi_{(t, 1)} = \cos(t)$$

Suppose $\Phi_{t_1} = \Phi_{t_2}$ for two different integer positions $t_1 \ne t_2$. Then in particular:
$$\sin(t_1) = \sin(t_2) \quad \text{and} \quad \cos(t_1) = \cos(t_2)$$

From trigonometry, for both sine and cosine of two real numbers to be equal, their difference must be an integer multiple of $2\pi$:
$$t_1 - t_2 = 2\pi m \quad \text{for some integer } m \in \mathbb{Z}$$

Since $t_1 \ne t_2$, we have $m \ne 0$. Solving for $\pi$:
$$\pi = \frac{t_1 - t_2}{2m}$$

Because $t_1, t_2,$ and $m$ are integers, this would mean $\pi$ is a **rational number** (a ratio of two integers). 
However, **$\pi$ is irrational**. 

## 03 `RoPE` (Rotary Positional Embedding) 

### 01 Intro

Given a feature vector with two features $x^{(1)}_t$ and $x^{(2)}_t$ at position $t$ in the sequence, the RoPE
positional embedding is defined as:
$$
\begin{aligned}
RoPE(x^{(1)}_t ,x^{(2)}_t ,t) 

&=
\begin{pmatrix}
\cos(t \theta) - \sin(t \theta) \\
\sin(t \theta) + \cos(t \theta)
\end{pmatrix}

\begin{pmatrix}
x^{(1)}_t \\
x^{(2)}_t
\end{pmatrix} \\

&=
\begin{pmatrix}
x^{(1)}_t \cos(t \theta) - x^{(2)}_t \sin(t \theta) \\
x^{(1)}_t \sin(t \theta) + x^{(2)}_t \cos(t \theta)
\end{pmatrix}
\end{aligned}
$$
where $\theta_t$ is a function of the position $t$.

For a d dimensional feature, RoPE is applied to each pair of features with an angle $\theta_i$ defined as
$
\theta_i = 10000^{-2(i-1)/d}, \quad i \in \{1,2,...,d/2\}.
$

$$
\begin{pmatrix}
\cos(t \theta_1) & -\sin(t \theta_1) & 0 & 0 & \dots & 0 & 0\\
\sin(t \theta_1) & \cos(t \theta_1) & 0 & 0 & \dots & 0 & 0\\
0 & 0 & \cos(t \theta_2) & -\sin(t \theta_2) & \dots & 0 & 0\\
0 & 0 & \sin(t \theta_2) & \cos(t \theta_2) & \dots & 0 & 0\\
\vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots\\
0 & 0 & 0 & 0 & \dots & \cos(t \theta_{d/2}) & -\sin(t \theta_{d/2})\\
0 & 0 & 0 & 0 & \dots & \sin(t \theta_{d/2}) & \cos(t \theta_{d/2})
\end{pmatrix}
\begin{pmatrix}
x_t^{(1)} \\
x_t^{(2)} \\
x_t^{(3)} \\
x_t^{(4)} \\
\vdots \\
x_t^{(d-1)} \\
x_t^{(d)}
\end{pmatrix}
\tag{3}
$$

### 02 Question (i)

i. (2 points) Using the rotation interpretation, RoPE operation can be viewed as rotation of the complex number $x^{(1)}_t + ix^{(2)}_t$ by an angle $t\theta$. Recall that this corresponds to multiplication by
$e^{it\theta} = \cos t\theta + i\sin t\theta$.

For higher dimensional feature vectors, this interpretation allows us to compute Equation 3 more
efficiently. Specifically, we can rewrite the RoPE operation as an element-wise multiplication
(denoted by $\odot$) of two vectors as follows:

$$
\begin{pmatrix}
\cos(t\theta_1) + i\sin(t\theta_1) \\
\cos(t\theta_2) + i\sin(t\theta_2) \\
\vdots \\
\cos(t\theta_{d/2}) + i\sin(t\theta_{d/2})
\end{pmatrix}

\odot
\begin{pmatrix}
x_t^{(1)} + ix_t^{(2)} \\
x_t^{(3)} + ix_t^{(4)} \\
\vdots \\
x_t^{(d-1)} + ix_t^{(d)}
\end{pmatrix}

\tag{4}
$$

Show that the elements of the vector in Equation 3 can be obtained from Equation 4. Note
that some additional operations like reshaping are necessary to make the two expressions equal
but you do not need to provide a detailed derivation for full points.

Let's show how the elements of the vector in Equation 3 can be obtained from Equation 4.

$$ 
(\cos(t\theta_1) + i\sin(t\theta_1)) (x_t^{(1)} + ix_t^{(2)}) = (\cos(t\theta_1)x_t^{(1)} - \sin(t\theta_1)x_t^{(2)}) + i(\sin(t\theta_1)x_t^{(1)} + \cos(t\theta_1)x_t^{(2)}) = z_1

$$

We now need to extract the real and imaginary parts of the complex vector obtained from Equation 4 to match the elements of the vector in Equation 3.

$$
\begin{pmatrix}
\text{Re}\left[z_1\right] \\
\text{Im}\left[z_1\right] \\
\dots \\
\text{Re}\left[z_{d/2}\right] \\
\text{Im}\left[z_{d/2}\right] \\
\end{pmatrix}
$$


To do this in PyTorch we may use `torch.view_as_real(z).flatten(-2)`

In [5]:
# 1. Setup a small example with dimension d = 4, at position t = 3
d = 4
t = 3

# Input vector x_t of shape (d,)
x_t = torch.tensor([1.0, 2.0, 3.0, 4.0])

# Frequency angles theta for each pair (d/2 = 2 frequencies)
theta = torch.tensor([0.5, 0.1])
angles = t * theta  # t * theta_k

In [9]:
x_t

tensor([1., 2., 3., 4.])

In [6]:
# =====================================================================
# METHOD 1: Equation 3 (Block-Diagonal Matrix Multiplication)
# =====================================================================
# Build the 4x4 rotation matrix R_t
c1, s1 = torch.cos(angles[0]), torch.sin(angles[0])
c2, s2 = torch.cos(angles[1]), torch.sin(angles[1])

R_t = torch.tensor([
    [ c1, -s1,   0,   0],
    [ s1,  c1,   0,   0],
    [  0,   0,  c2, -s2],
    [  0,   0,  s2,  c2],
])

# Matrix multiplication: R_t @ x_t
out_matrix = R_t @ x_t

In [7]:
R_t

tensor([[ 0.0707, -0.9975,  0.0000,  0.0000],
        [ 0.9975,  0.0707,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.9553, -0.2955],
        [ 0.0000,  0.0000,  0.2955,  0.9553]])

In [8]:
out_matrix

tensor([-1.9243,  1.1390,  1.6839,  4.7079])

In [10]:
R_t[0, :] @ x_t

tensor(-1.9243)

In [12]:
# =====================================================================
# METHOD 2: Equation 4 (Complex Element-wise Multiplication in PyTorch)
# =====================================================================
# Step A: Pair consecutive coordinates into shape (d/2, 2)
# [x1, x2, x3, x4] -> [[x1, x2], [x3, x4]]
x_paired = x_t.reshape(-1, 2)

# Step B: View as complex numbers: [x1 + i*x2,  x3 + i*x4]
x_complex = torch.view_as_complex(x_paired)

# Step C: Form the complex rotation vector: [cos(t*theta) + i*sin(t*theta)]
rot_complex = torch.polar(torch.ones_like(angles), angles)
# (or equivalently: torch.cos(angles) + 1j * torch.sin(angles))

# Step D: Element-wise multiplication (Equation 4)
out_complex = x_complex * rot_complex

# Step E: Reshape back to real: unpack real/imag and flatten back to shape (d,)
out_rope = torch.view_as_real(out_complex).flatten()

In [13]:
x_paired

tensor([[1., 2.],
        [3., 4.]])

In [14]:
x_complex

tensor([1.+2.j, 3.+4.j])

In [15]:
rot_complex

tensor([0.0707+0.9975j, 0.9553+0.2955j])

In [17]:
out_matrix

tensor([-1.9243,  1.1390,  1.6839,  4.7079])

In [16]:
out_complex

tensor([-1.9243+1.1390j,  1.6839+4.7079j])

In [18]:
torch.view_as_real(out_complex)

tensor([[-1.9243,  1.1390],
        [ 1.6839,  4.7079]])

In [19]:
torch.view_as_real(out_complex).flatten()

tensor([-1.9243,  1.1390,  1.6839,  4.7079])

### 03 Question (ii)

ii. (1 point) Relative Embeddings. Now we will show that the dot product of the RoPE embeddings of two vectors at positions $t_1$ and $t_2$ depends on the relative position $t_1−t_2$ only.

For simiplicity, we will assume two dimensional feature vectors (eg. $[a,b]$) and work with their
complex number representations (eg. $a+ ib$).

Show that $\langle \text{RoPE}(z_1,t_1),\text{RoPE}(z_2,t_2)\rangle= \langle \text{RoPE}(z_1,t_1−t_2),\text{RoPE}(z_2,0)\rangle$ where $\langle·,·\rangle$ denotes the dot product and $\text{RoPE}(z,t)$ is the RoPE embedding of vector $z$ at position $t$.

(Hint: Dot product of vectors represented as complex numbers is given by $\langle z_1,z_2\rangle= \mathrm{Re}(z_1\bar{z_2})$.
For a complex number $z = a+ ib$ ($a,b\in\mathbb{R}$), $\mathrm{Re}(z) = a$ indicates the real component of $z$ and
$\bar{z}= a-ib$ is the complex conjugate of $z$.)

---

Let's introduce the new notation: $ R_t = e^{i\theta t} = \cos(\theta t) + i\sin(\theta t)$. Now let's show that $R_{t_1} \bar{R}_{t_2} = R_{t_1-t_2}$.

We have:
$$
R_{t_1} \bar{R}_{t_2} = e^{i\theta t_1} \cdot e^{-i\theta t_2} = e^{i\theta (t_1-t_2)} = R_{t_1-t_2}.
$$

What is $\text{RoPE}(z_2, 0)$? 
$$
\text{RoPE}(z_2, 0) = z_2 \cdot R_0 = z_2 \cdot 1 = z_2.
$$

Finally, let's show that the dot product depends only on the relative position:
$$
\begin{aligned}
\langle \text{RoPE}(z_1,t_1),\text{RoPE}(z_2,t_2)\rangle 
&= \mathrm{Re}\big((R_{t_1} z_1 ) \overline{(R_{t_2} z_2 )}\big) \\
&= \mathrm{Re}\big(R_{t_1} \bar{R}_{t_2} z_1 \bar{z_2} \big) \\
&= \mathrm{Re}\big(R_{t_1-t_2} z_1 \bar{z_2} \big) \\
&= \langle \text{RoPE}(z_1,t_1-t_2),\text{RoPE}(z_2,0)\rangle.
\end{aligned}
$$
